# Building an AI-powered multimodal RAG system with Docling
*Using IBM Granite vision, text-based embeddings and generative AI models*


## Lab 3: Multimodal RAG with Visual Grounding

In Lab 1 you converted documents with Docling, and in Lab 2 you split them into chunks. In this lab you put both to work in a retrieval-augmented generation (RAG) system that answers questions about a PDF and shows where each answer came from.

Showing the source matters because a generated answer is hard to trust on its own. When the system highlights the passage, table, or picture it used on the original page, you can check the answer in a few seconds instead of searching the document yourself.

## What you'll build

A RAG system that:

1. Indexes the text, tables, and pictures in a PDF
2. Describes each picture with a vision model, so pictures can be found by search
3. Answers questions with a Granite language model
4. Draws the retrieved passages, tables, and pictures on the original pages

---


## Background

### What is multimodal RAG?

[Retrieval-augmented generation (RAG)](https://www.ibm.com/think/topics/retrieval-augmented-generation) connects a large language model (LLM) to information it wasn't trained on, without [fine-tuning](https://www.ibm.com/think/topics/rag-vs-fine-tuning). Most RAG systems only index text, but real documents also contain tables (structured data, financial figures) and pictures (charts, diagrams, photos).

Multimodal RAG brings those into the knowledge base too. In this lab, tables are indexed as Markdown, and pictures are indexed through text descriptions written by a [multimodal model](https://www.ibm.com/think/topics/multimodal-ai).

### Visual grounding

Docling records where every item in a document came from: the page number and a bounding box on that page. Visual grounding uses that record to show the source of each retrieved chunk by:
- Drawing a box around it on the page image
- Labeling tables and pictures (TABLE, IMAGE)
- Using a different color for each content type

You'll use IBM Granite models for the vision and language steps.

---


## Lab overview

In this lab you will:

1. Configure Docling to keep page images and item locations
2. Chunk text and tables, and describe pictures with a vision model
3. Store everything in a vector database along with its source location
4. Retrieve chunks for a question and highlight them on the page
5. Generate an answer from the retrieved chunks

### Technologies

- [Docling](https://docling-project.github.io/docling/) to convert and chunk documents
- [LangChain](https://langchain.com) to connect retrieval and generation
- [IBM Granite](https://www.ibm.com/granite/) models for embeddings, picture descriptions, and answers
- [ChromaDB](https://docs.trychroma.com/docs) as the vector database

---

## Prerequisites

- Python 3.11 or later
- [Ollama](https://ollama.com) installed and running
- A basic understanding of embeddings and vector databases

## Setting up the environment

Ensure you are running Python 3.11 or later in a freshly created virtual environment.

In [ ]:
import sys
assert sys.version_info >= (3, 11) and sys.version_info < (3, 14), "Use Python 3.11, 3.12, or 3.13 to run this notebook."

## Install Dependencies

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    transformers \
    pillow \
    langchain \
    langchain-classic \
    langchain_community \
    "langchain_huggingface[full]" \
    langchain_chroma chromadb \
    langchain-openai\
    docling \
    matplotlib
! echo "::endgroup::"

## Import Required Libraries and Configure Logging


### Logging Configuration

Set the log level to INFO to see what Docling is doing. You can skip this cell if you prefer less output.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

### Required Libraries

Import the modules used in this lab:

In [ ]:
# import itertools
import tempfile
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import PIL.Image
import PIL.ImageOps
from PIL import ImageDraw
from IPython.display import display

# Docling imports for document processing and visual grounding
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.document import DoclingDocument
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types.doc.document import RefItem

# LangChain imports for RAG pipeline
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStore
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Model imports
from transformers import AutoTokenizer

---

## Selecting and loading the AI models

### The three models

This lab uses three models:

1. An **embeddings model** turns text into vectors, so chunks with similar meaning can be found. It embeds text chunks, tables, and picture descriptions alike.
2. A **vision model** writes a text description of each picture in the document, so pictures can be retrieved like text.
3. A **language model** writes the answer from the retrieved chunks.

### Load the Granite Embeddings Model

We'll use `granite-embedding-small-english-r2` from the [Granite Embedding models](https://huggingface.co/collections/ibm-granite/granite-embedding-models-6750b30c802c1926a35550bb). It's an English model with 47M parameters, small enough to run on a CPU, and it accepts up to 8,192 tokens. This lab keeps chunks to 512 tokens.

To use a different embeddings model, see [this Embeddings Model recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Embeddings_Models.ipynb).

### Your turn: pick an embeddings model

The embeddings model controls *what the vector database considers similar*. A better embedder returns more relevant chunks for a given query, which directly improves answer quality.

<details>
<summary>What should I try?</summary>

Alternatives to try:
- `"ibm-granite/granite-embedding-small-english-r2"` — small, fast, English-only (the default).
- `"ibm-granite/granite-embedding-311m-multilingual-r2"` — larger and multilingual, for non-English documents.
- `"sentence-transformers/all-MiniLM-L6-v2"` — a classic general-purpose baseline.

After changing this, you must re-run the vector database cells below, because embeddings from different models can't be mixed.

</details>


In [ ]:
# TODO: try swapping the embeddings model below and re-running retrieval to see how it changes results.
embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"  # <-- change me
embeddings_model = HuggingFaceEmbeddings(
    model_name=embeddings_model_path,
)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)
print(f"Embeddings model loaded: {embeddings_model_path}")


### Load the Granite Vision Model

The vision model describes the pictures in the document. Those descriptions are what make charts and figures searchable.

This notebook runs Granite Vision locally with Ollama, so no API keys are needed and your documents stay on your machine. Pull the model first:

```shell
ollama pull granite3.2-vision:2b
```

In [ ]:
# Vision model is set up later in the chunking cell via Ollama
# Model: granite3.2-vision:2b at http://localhost:11434/v1
print("Vision model will be loaded via Ollama when processing images")

### Load the Granite Language Model

The language model writes the final answer from the retrieved chunks. We use Granite 4.2 8B, an instruction-tuned, Apache 2.0 licensed model that balances answer quality and speed. Pull it first:

```shell
ollama pull granite4.2:8b
```

In [ ]:
model_path = "granite4.2:8b"
from langchain_openai import ChatOpenAI
model = ChatOpenAI(
    model_name=model_path,
    api_key="none",
    base_url="http://localhost:11434/v1",  # local Ollama
    temperature=0.01,
)

print(f"Language model loaded: {model_path}")


---


## Convert documents for visual grounding

### What visual grounding needs from conversion

To highlight sources later, the conversion has to keep:

1. **Page images**, to draw the highlights on
2. **Item locations** (page number and bounding box), which Docling records by default
3. **The converted document**, saved to disk so it can be reloaded when you visualize a result

The next cells configure Docling for this.

### Optional: Run Docling on a hosted service

Docling runs on this machine (or Colab runtime) by default. You can instead send the conversions to a Docling service: [Docling for IBM watsonx](https://www.ibm.com/products/docling), IBM's managed Docling service, or any docling-serve instance, such as the one you start in the Docling as a Service lab. The service returns the same `ConversionResult` as a local conversion, so the rest of the notebook works unchanged.

To use a service, set these as environment variables, in a `.env` file, or as Colab secrets before running the next cell:

- `DOCLING_SERVICE_URL`: the service endpoint URL
- `DOCLING_SERVICE_API_KEY`: your API key (leave it unset for a docling-serve that does not require one)

If `DOCLING_SERVICE_URL` is not set, Docling runs locally.

In [ ]:
from ibm_granite_community.notebook_utils import get_env_var

# Each setting comes from the environment, a .env file, or Colab secrets, and is empty if unset
DOCLING_SERVICE_URL = get_env_var("DOCLING_SERVICE_URL", "")
DOCLING_SERVICE_API_KEY = get_env_var("DOCLING_SERVICE_API_KEY", "")
USE_DOCLING_SERVICE = bool(DOCLING_SERVICE_URL)

if USE_DOCLING_SERVICE:
    from docling.datamodel.service.options import ConvertDocumentsOptions
    from docling.service_client import DoclingServiceClient

    print(f"Docling conversions will run on the service at {DOCLING_SERVICE_URL}")
else:
    print("Docling conversions will run locally")

In [ ]:
# Configure converter with visual grounding capabilities
if USE_DOCLING_SERVICE:
    converter = DoclingServiceClient(
        url=DOCLING_SERVICE_URL,
        api_key=DOCLING_SERVICE_API_KEY,
        options=ConvertDocumentsOptions(
            do_ocr=False,  # Set to True if your PDFs contain scanned images
            include_images=True,  # Return images from documents
            include_page_images=True,  # CRITICAL: Return page images for visual grounding
            image_export_mode="embedded",  # Keep the images inside the returned document
        ),
    )
else:
    pdf_pipeline_options = PdfPipelineOptions(
        do_ocr=False,  # Set to True if your PDFs contain scanned images
        generate_picture_images=True,  # Extract images from documents
        generate_page_images=True,  # CRITICAL: Generate page images for visual grounding
    )

    format_options = {
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_pipeline_options),
    }

    converter = DocumentConverter(format_options=format_options)

print("Document converter configured with visual grounding support")

### Create a Document Store for Visual Grounding

The document store keeps each converted document as JSON, keyed by the document's hash, so the visualization code can reload it later:

In [ ]:
# Create document store for visual grounding
doc_store = {}
doc_store_root = Path(mkdtemp())
print(f"Document store created at: {doc_store_root}")

### Convert the documents

Convert each source and save it to the document store:

### Your turn: swap in your own document

This RAG system works on any PDF. Try replacing the URL with a document you care about, such as an annual report, a research paper, or a product manual, and run the rest of the notebook on it.

<details>
<summary>What should I try?</summary>

Good alternatives:
- **Another annual report** — same domain, so you can compare results head-to-head.
- **A research paper** (arXiv PDF URL) — tests dense technical content with equations and figures.
- **A local file** — replace the URL with a path like `"/path/to/my.pdf"`.

A text-only PDF works too, but it won't exercise the table and picture paths.

</details>


In [ ]:
# TODO: try swapping the sources URL below to point at a different PDF (your own doc works too).
sources = [
    "https://github.com/ibm-granite-community/docling-workshop/releases/download/fixtures/v1/AR_2020_WEB2.pdf",  # <-- change me
    # Add more document URLs here as needed
]

conversions = {}

print("Starting document conversion with visual grounding support...")
for source in sources:
    print(f"\n Processing: {source}")

    # Convert document
    result = converter.convert(source=source)
    docling_document = result.document
    conversions[source] = docling_document

    # Save document to store for visual grounding
    # The binary hash ensures unique identification
    origin_hash = str(docling_document.origin.binary_hash)
    file_path = doc_store_root / f"{origin_hash}.json"
    docling_document.save_as_json(file_path)
    doc_store[origin_hash] = file_path

    print("Document converted and saved")
    print(f"  - Document ID: {origin_hash}")
    print(f"  - Pages: {len(docling_document.pages)}")
    print(f"  - Tables: {len(docling_document.tables)}")
    print(f"  - Images: {len(docling_document.pictures)}")


## Chunk the documents

### Keep the source location with every chunk

Each chunk keeps the page numbers and bounding boxes of the items it came from, a reference to its source document, and its content type (text, table, or picture). The visualization code uses these to draw the highlights.

### Chunk text and tables, describe pictures

`HybridChunker` splits the text and tables into chunks and keeps each chunk's section headings. Each chunk is embedded with those headings in front (`chunker.contextualize(chunk)`), which tells the embeddings model what the chunk is about. `MarkdownTableSerializer` writes tables as Markdown. Pictures are handled separately: the Granite vision model describes each one, and the description becomes its own chunk.

Text and table chunks store their Docling metadata (`dl_meta`), including page numbers and bounding boxes. Picture chunks store a reference to the picture instead.

Describing the pictures can take a while, because each one is sent to the vision model separately.

In [ ]:
import base64
import io

from docling.chunking import DocMeta
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer

# Helper to encode images for the vision model
def encode_image(image: PIL.Image.Image, format: str = "png") -> str:
    """Encode image to base64 for vision model processing"""
    image = PIL.ImageOps.exif_transpose(image) or image
    image = image.convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format)
    encoding = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/{format};base64,{encoding}"

# Set up vision model via Ollama for image description
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

vision_model = ChatOpenAI(
    model_name="granite3.2-vision:2b",
    api_key="none",
    base_url="http://localhost:11434/v1",
    temperature=0.01,
    max_tokens=512,  # keep descriptions within the 512-token chunk size
)
image_prompt = "Give a detailed description of what is depicted in the image"

class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            table_serializer=MarkdownTableSerializer(),
        )

doc_id = 0
chunks: list[Document] = []
num_text_chunks = 0
num_table_chunks = 0
num_picture_chunks = 0

# max_tokens is set explicitly so behavior does not depend on the
# wrapped HF tokenizer's model_max_length, which varies across transformers versions.
chunker_tokenizer = HuggingFaceTokenizer(
    tokenizer=embeddings_tokenizer,
    max_tokens=512,
)

print("\nProcessing all content with visual grounding metadata...")
for source, docling_document in conversions.items():
    chunker = HybridChunker(
        tokenizer=chunker_tokenizer,
        serializer_provider=MDTableSerializerProvider(),
    )

    # Process text and table chunks via the unified chunker
    for chunk in chunker.chunk(docling_document):
        items = chunk.meta.doc_items
        refs = " ".join(map(lambda item: item.get_ref().cref, items))

        if "#/tables" in refs:
            num_table_chunks += 1
            item_type = "table"
        else:
            num_text_chunks += 1
            item_type = "text"

        text = chunker.contextualize(chunk=chunk)

        document = Document(
            page_content=text,
            metadata={
                "doc_id": (doc_id := doc_id + 1),
                "source": source,
                "ref": refs,
                "dl_meta": chunk.meta.model_dump_json(),
                "item_type": item_type,
                "picture_ref": "",
                "origin_hash": str(docling_document.origin.binary_hash),
            },
        )
        chunks.append(document)

    # Process pictures separately using Ollama vision model
    for picture in docling_document.pictures:
        ref = picture.get_ref().cref
        image = picture.get_image(docling_document)
        if image:
            print(f"  Processing image: {ref}")
            # Call Ollama vision model with image
            msg = HumanMessage(content=[
                {"type": "image_url", "image_url": {"url": encode_image(image)}},
                {"type": "text", "text": image_prompt},
            ])
            response = vision_model.invoke([msg])
            text = response.content
            num_picture_chunks += 1

            document = Document(
                page_content=text,
                metadata={
                    "doc_id": (doc_id := doc_id + 1),
                    "source": source,
                    "ref": ref,
                    "dl_meta": "{}",
                    "item_type": "picture",
                    "picture_ref": ref,
                    "origin_hash": str(docling_document.origin.binary_hash),
                },
            )
            chunks.append(document)

print(f"\nCreated {len(chunks)} chunks with visual grounding metadata")
print(f"  {num_text_chunks} text chunks")
print(f"  {num_table_chunks} table chunks")
print(f"  {num_picture_chunks} picture chunks")

### Display Sample Processed Documents

Print a few chunks of each type:

In [ ]:
print("\nSample processed documents:")
print("=" * 80)

# Show sample text chunks
print("\nTEXT CHUNK EXAMPLES:")
print("-" * 80)
c = 0
for chunk in chunks:  # Show first 3 text chunks
    if chunk.metadata["item_type"] != "text":
        continue
    c += 1
    print(f"\nText Chunk {c}:")
    print(f"  Document ID: {chunk.metadata['doc_id']}")
    print(f"  Source: {chunk.metadata['source'].split('/')[-1]}")  # Just filename
    print(f"  Reference: {chunk.metadata['ref']}")
    print(f"  Has visual grounding: {'dl_meta' in chunk.metadata}")
    print(f"  Content preview:")
    print(f"    {chunk.page_content[:250]}...")
    print("-" * 40)
    if c >= 3:
        break

# Show sample tables
print("\n\nTABLE EXAMPLES:")
print("-" * 80)
c = 0
for chunk in chunks:  # Show first 3 tables
    if chunk.metadata["item_type"] != "table":
        continue
    c += 1
    print(f"\nTable {c}:")
    print(f"  Document ID: {chunk.metadata['doc_id']}")
    print(f"  Reference: {chunk.metadata['ref']}")
    print(f"  Content preview (Markdown format):")
    # Show first few lines of the table
    table_lines = chunk.page_content.split('\n')[:8]
    for line in table_lines:
        print(f"    {line}")
    print("-" * 40)
    if c >= 3:
        break

# Show sample images with descriptions
print("\n\nIMAGE EXAMPLES WITH AI-GENERATED DESCRIPTIONS:")
print("-" * 80)
c = 0
for chunk in chunks:  # Show first 3 images
    if chunk.metadata["item_type"] != "picture":
        continue
    c += 1
    print(f"\nImage {c}:")
    print(f"  Document ID: {chunk.metadata['doc_id']}")
    print(f"  Reference: {chunk.metadata['ref']}")
    print(f"  AI-Generated Description:")
    # Wrap the description for better readability
    import textwrap
    wrapped_text = textwrap.fill(chunk.page_content, width=70, initial_indent="    ", subsequent_indent="    ")
    print(wrapped_text)

    # Display the actual image
    source = chunk.metadata['source']
    ref = chunk.metadata['picture_ref']
    docling_document = conversions[source]
    picture = RefItem(cref=ref).resolve(docling_document)
    image = picture.get_image(docling_document)
    if image:
        print(f"\n  Original Image:")
        # Resize image for display if too large
        display_image = image.copy()
        max_width = 600
        if display_image.width > max_width:
            ratio = max_width / display_image.width
            new_height = int(display_image.height * ratio)
            display_image = display_image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
        display(display_image)

    print("-" * 40)
    if c >= 3:
        break


## Visual grounding

### Show where a chunk came from

The function below takes a retrieved chunk, reloads its source document, and draws a box around each item the chunk came from on the page images. Tables and pictures get a label and a thicker box.

In [ ]:
def visualize_chunk_grounding(chunk, doc_store, highlight_color="blue"):
    """
    Visualize where a chunk comes from in the original document.

    This function:
    1. Loads the original document from the store
    2. Finds the pages containing the chunk content
    3. Draws bounding boxes around the source regions
    4. Displays the highlighted pages

    Args:
        chunk: LangChain Document with visual grounding metadata
        doc_store: Dictionary mapping document hashes to file paths
        highlight_color: Color for highlighting (blue, green, red, etc.)

    Returns:
        Dictionary of page images with highlights
    """
    origin_hash = chunk.metadata.get("origin_hash")
    if not origin_hash:
        print("No origin hash found in metadata")
        return None

    # Load the full document from store
    dl_doc = DoclingDocument.load_from_json(doc_store.get(origin_hash))
    print(f"Visualizing source location for chunk {chunk.metadata.get('doc_id', 'Unknown')}")

    page_images = {}
    item_type = chunk.metadata.get("item_type", "text")

    # For pictures processed outside the chunker, resolve provenance from the document directly
    if item_type == "picture" and "picture_ref" in chunk.metadata and chunk.metadata["picture_ref"]:
        try:
            picture = RefItem(cref=chunk.metadata["picture_ref"]).resolve(dl_doc)
            if hasattr(picture, 'prov') and picture.prov:
                for prov in picture.prov:
                    page_no = prov.page_no
                    if page_no in dl_doc.pages:
                        page = dl_doc.pages[page_no]
                        if hasattr(page, 'image') and page.image:
                            if page_no not in page_images:
                                img = page.image.pil_image.copy()
                                page_images[page_no] = {
                                    'image': img, 'page': page,
                                    'draw': ImageDraw.Draw(img)
                                }
                            draw = page_images[page_no]['draw']
                            bbox = prov.bbox.to_top_left_origin(page_height=page.size.height)
                            bbox = bbox.normalized(page.size)
                            l = int(bbox.l * page_images[page_no]['image'].width)
                            r = int(bbox.r * page_images[page_no]['image'].width)
                            t = int(bbox.t * page_images[page_no]['image'].height)
                            b = int(bbox.b * page_images[page_no]['image'].height)
                            draw.rectangle([l, t, r, b], outline=highlight_color, width=4)
                            draw.text((l, t - 20), "IMAGE", fill=highlight_color)
        except Exception as e:
            print(f"Error processing picture metadata: {e}")
            return None

    elif "dl_meta" in chunk.metadata and chunk.metadata["dl_meta"]:
        # For text and table chunks produced by the chunker, use dl_meta
        try:
            meta = DocMeta.model_validate_json(chunk.metadata["dl_meta"])

            for doc_item in meta.doc_items:
                if hasattr(doc_item, 'prov') and doc_item.prov:
                    for prov in doc_item.prov:
                        page_no = prov.page_no

                        if page_no not in page_images:
                            if page_no in dl_doc.pages:
                                page = dl_doc.pages[page_no]
                                if hasattr(page, 'image') and page.image:
                                    img = page.image.pil_image.copy()
                                    page_images[page_no] = {
                                        'image': img, 'page': page,
                                        'draw': ImageDraw.Draw(img)
                                    }

                        if page_no in page_images:
                            page_data = page_images[page_no]
                            page = page_data['page']
                            draw = page_data['draw']

                            bbox = prov.bbox.to_top_left_origin(page_height=page.size.height)
                            bbox = bbox.normalized(page.size)

                            l = int(bbox.l * page_data['image'].width)
                            r = int(bbox.r * page_data['image'].width)
                            t = int(bbox.t * page_data['image'].height)
                            b = int(bbox.b * page_data['image'].height)

                            if item_type == "table":
                                draw.rectangle([l, t, r, b], outline=highlight_color, width=3)
                                draw.text((l, t - 20), "TABLE", fill=highlight_color)
                            else:
                                draw.rectangle([l, t, r, b], outline=highlight_color, width=2)
        except Exception as e:
            print(f"Error processing chunk metadata: {e}")
            return None
    else:
        print("No visual grounding metadata available for this chunk")
        return None

    # Display highlighted pages (page_no is already 1-indexed in docling provenance)
    for page_no, page_data in sorted(page_images.items()):
        plt.figure(figsize=(12, 16))
        plt.imshow(page_data['image'])
        plt.axis('off')

        title_map = {"picture": "Image Location", "table": "Table Location"}
        title = title_map.get(item_type, "Text Location")
        plt.title(f'{title} - Page {page_no}', fontsize=16)
        plt.tight_layout()
        plt.show()

    return page_images

## Populate the Vector Database

### What goes into the vector database

The vector database stores an embedding for each chunk along with its metadata, including the source location used for highlighting. Text chunks, Markdown tables, and picture descriptions are all embedded the same way, so one search covers all three.

### Choose and Configure Vector Database

We'll use ChromaDB. For other vector database options, see [this Vector Store recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Vector_Stores.ipynb).

In [ ]:
# Initialize ChromaDB vector store
vector_db: VectorStore = Chroma(embedding_function=embeddings_model)


### Add Documents to Vector Database

Add the text, table, and picture chunks to the vector database:

In [ ]:
print("\nAdding documents to vector database...")
documents = chunks
ids = vector_db.add_documents(documents)
print(f"Successfully added {len(ids)} documents to the vector database")

## Test retrieval

Before adding the language model, check that retrieval finds relevant chunks. If it doesn't, the answers won't be good either, so this is the place to catch problems.

### Your turn: preview what the retriever finds

This cell only runs retrieval (no generation). It shows whether the vector database returns the right chunks for your question before you run the full RAG pipeline.

<details>
<summary>What should I try?</summary>

Useful experiments:
- Target a **table**, for example `"What were total 2020 expenses?"`, and check whether the results include `Type: Table (Markdown)`.
- Target a **picture**, for example `"Show me the food distribution map."`, and check for `Type: AI-Generated Image Description`.
- Try a vague query, such as `"Tell me about volunteers."`, to see whether the retriever still finds relevant content.

If the retriever misses obvious content, the RAG answer below will also miss it. This cell is the place to diagnose retrieval issues.

</details>


In [ ]:
# TODO: try changing test_query to test retrieval before running the full RAG pipeline.
test_query = "How much was spent on food distribution relative to the amount of food distributed?"  # <-- change me

print(f"\nTesting retrieval for query: '{test_query}'")
print("=" * 80)

# Retrieve relevant documents
retrieved_docs = vector_db.as_retriever().invoke(test_query)

# Display retrieved documents
for i, doc in enumerate(retrieved_docs):
    print(f"\nRetrieved Document {i+1}:")

    # Determine content type
    item_type = doc.metadata.get('item_type', 'text')
    if item_type == 'picture':
        content_type = "AI-Generated Image Description"
    elif item_type == 'table':
        content_type = "Table (Markdown)"
    else:
        content_type = "Text Chunk"

    print(f"Type: {content_type}")
    print(f"Content preview: {doc.page_content[:200]}...")
    print(f"Source: {doc.metadata['source'].split('/')[-1]}")


## RAG with visual grounding

The function below:
1. Retrieves the chunks most relevant to a question
2. Highlights where each one came from
3. Generates an answer from them

### Highlight styles

Each content type is highlighted differently, so you can see at a glance what the answer drew on:

- **Text**: thin blue box
- **Tables**: thicker green box labeled TABLE
- **Pictures**: thicker red box labeled IMAGE

In [ ]:
def rag_with_visual_grounding(question, vector_db, doc_store, model, top_k=5):
    """
    Perform RAG with visual grounding of results.

    This function:
    1. Retrieves relevant chunks from the vector database
    2. Visualizes where each chunk comes from in the original document
    3. Generates a response using the retrieved context

    Args:
        question: User's query
        vector_db: Vector database with embedded documents
        doc_store: Document store for visual grounding
        model: Language model for response generation
        top_k: Number of chunks to retrieve

    Returns:
        Tuple of (outputs, relevant_chunks)
    """
    print(f"\nQuestion: {question}")
    print("=" * 80)

    # Step 1: Retrieve relevant chunks
    print(f"\nRetrieving top {top_k} relevant chunks...")
    retriever = vector_db.as_retriever(search_kwargs={"k": top_k})
    relevant_chunks = retriever.invoke(question)

    print(f"Found {len(relevant_chunks)} relevant chunks")

    # Step 2: Visualize each chunk's source location
    print("\nVisualizing source locations:")

    for i, chunk in enumerate(relevant_chunks):
        print(f"\n--- Result {i+1} ---")

        # Determine content type
        item_type = chunk.metadata.get('item_type', 'text')
        if item_type == 'picture':
            content_type = "AI-Generated Image Description"
            color = 'red'
        elif item_type == 'table':
            content_type = "Table (Markdown)"
            color = 'green'
        else:
            content_type = "Text Chunk"
            color = 'blue'

        print(f"Content type: {content_type}")
        print(f"Text preview: {chunk.page_content[:200]}...")
        print(f"Source: {chunk.metadata.get('source', 'Unknown').split('/')[-1]}")

        # Show visual grounding if available
        if "dl_meta" in chunk.metadata or "prov_data" in chunk.metadata:
            visualize_chunk_grounding(
                chunk,
                doc_store,
                highlight_color=color
            )
        else:
            print("  (No visual grounding available for this chunk)")

    # Step 3: Create RAG pipeline for response generation
    print("\nGenerating response with LLM...")

    # Put the retrieved chunks in the system message; Ollama applies Granite's chat template
    prompt_template = ChatPromptTemplate.from_messages([
        ("system",
         "Write the response to the user's input by strictly aligning with the facts in the provided documents. "
         "If the information needed to answer the question is not available in the documents, "
         "inform the user that the question cannot be answered based on the available data.\n\n"
         "Documents:\n{context}"),
        ("human", "{input}"),
    ])

    # Document prompt template
    document_prompt_template = PromptTemplate.from_template(template="[Document {doc_id}]\n{page_content}")

    # Create chains
    combine_docs_chain = create_stuff_documents_chain(
        llm=model,
        prompt=prompt_template,
        document_prompt=document_prompt_template,
        document_separator="\n\n",
    )

    rag_chain = create_retrieval_chain(
        retriever=retriever,
        combine_docs_chain=combine_docs_chain,
    )

    # Generate response
    outputs = rag_chain.invoke({"input": question})

    print("\nGenerated Answer:")
    print("=" * 80)
    print(outputs['answer'])
    print("=" * 80)

    return outputs, relevant_chunks

## Ask a question

### Your turn: ask the full RAG pipeline

This cell runs the whole pipeline: the local Ollama model answers your question from the retrieved chunks, and the page regions it used are highlighted. Try your own question.

<details>
<summary>What should I try?</summary>

Ideas for the food bank annual report:
- `"Who are the board members?"` — tests name/list extraction
- `"What percentage of donations went to programs?"` — tests arithmetic/table reading
- `"Summarize the mission statement."` — tests text synthesis
- `"What is shown on page 3?"` — tests visual grounding directly

Watch both the generated answer and the highlighted page regions beneath it.

</details>


In [ ]:
# TODO: try changing main_query to ask different questions about the document (or your own doc).
main_query = "How much was spent on food distribution relative to the amount of food distributed?"  # <-- change me
outputs, chunks = rag_with_visual_grounding(
    main_query,
    vector_db,
    doc_store,
    model,
)


## Summary

You built a RAG system that indexes the text, tables, and pictures in a PDF and shows where each retrieved chunk came from. Along the way you:

- Configured Docling to keep page images and item locations
- Chunked text and tables with `HybridChunker` and a Markdown table serializer
- Made pictures searchable with descriptions from a Granite vision model
- Drew retrieved chunks on the original pages, styled by content type
- Generated answers with Granite from the retrieved chunks

The highlights let you check an answer against its source in seconds. They don't stop the model from making mistakes, but they make mistakes much easier to spot.

---

## Next steps

- **Try your own documents**, especially ones with complex layouts, technical diagrams, or many pages of mixed content.
- **Adapt the picture prompt to your domain**, for example:
   ```python
   medical_prompt = "Describe this medical image, noting any abnormalities or key features"
   financial_prompt = "Analyze this financial chart, identifying trends and key data points"
   ```
- **Speed things up** by converting documents in batches, caching converted documents, and running the vision model on a GPU.

---

## Resources

- [Docling documentation](https://docling-project.github.io/docling/)
- [IBM Granite models](https://www.ibm.com/granite/)
- [LangChain documentation](https://python.langchain.com/)
- [ChromaDB documentation](https://docs.trychroma.com/docs)

Docling also works with [LlamaIndex](https://docs.llamaindex.ai/), [CrewAI](https://www.crewai.com/), and [Haystack](https://haystack.deepset.ai/), and can run as a service with [docling-serve](https://github.com/docling-project/docling-serve) or as an MCP server for AI agents. See the [Docling integrations documentation](https://docling-project.github.io/docling/integrations/) for guides.